# Predicting asset failure within 90 days

## Objective

Determine whether information available at inspection time can predict `failed_90d`, and quantify performance in a deployment-like setting.

The analysis deliberately separates **inspection-time features** from `parts_replaced`. The dictionary says that parts replaced are counted on the work order that *closed* the inspection record, so this field may only become known after the prediction point. Including it would create target leakage.

### Validation design

- Sort inspections by time; hold out the latest 20% as a final test period.
- Apply a 90-day label embargo: training ends 90 days before the test begins, so every training label could have matured by scoring time.
- Exclude the unique identifier `asset_tag` and leakage-prone `parts_replaced` from the deployable model.
- Use ROC AUC for ranking, average precision (AP) for rare-event usefulness, Brier score for probability error, and top-decile lift/recall for inspection triage.

This notebook is an experiment, not a production deployment. A final operating threshold requires failure/missed-failure costs and prospective validation.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load and validate the inputs

In [ ]:
data_path = Path("data.csv")
dictionary_path = Path("columns.csv")
if not data_path.exists() or not dictionary_path.exists():
    raise FileNotFoundError("Run this notebook from the directory containing data.csv and columns.csv")

df = pd.read_csv(data_path)
dictionary = pd.read_csv(dictionary_path)
df["inspected_at"] = pd.to_datetime(df["inspected_at"], errors="raise")

display(dictionary)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Inspection range: {df['inspected_at'].min()} to {df['inspected_at'].max()}")
display(df.head())

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
    "unique": df.nunique(dropna=False),
})
display(audit)

assert set(df["failed_90d"].unique()).issubset({0, 1}), "Target must be binary"
assert not df["failed_90d"].isna().any(), "Target contains missing values"

target_counts = df["failed_90d"].value_counts().sort_index().rename("count").to_frame()
target_counts["share"] = target_counts["count"] / len(df)
display(target_counts)
print(f"Overall failure prevalence: {df['failed_90d'].mean():.2%}")
print(f"Duplicate complete rows: {df.duplicated().sum():,}")
print(f"Repeated asset tags: {df['asset_tag'].duplicated().sum():,}")

## 2. Leakage and time-pattern audit

`asset_tag` is an identifier and every value is unique, so it cannot provide repeat-history signal. `parts_replaced` is much more serious: it is recorded when the work order closes and may encode actions taken after the inspection. We inspect it below but exclude it from all deployable models.

In [ ]:
parts_audit = pd.crosstab(df["parts_replaced"], df["failed_90d"], margins=True)
display(parts_audit)

monthly = (
    df.set_index("inspected_at")["failed_90d"]
      .resample("MS")
      .agg(["count", "sum", "mean"])
      .rename(columns={"mean": "failure_rate"})
)
display(monthly)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly["failure_rate"], marker="o")
ax.axhline(df["failed_90d"].mean(), color="black", linestyle="--", alpha=.6,
           label="overall prevalence")
ax.set(title="Failure prevalence changes substantially over time",
       xlabel="Inspection month", ylabel="90-day failure rate")
ax.legend()
plt.tight_layout()
plt.show()

The changing monthly rate makes a random train/test split inappropriate: it would mix past and future regimes. The final test therefore consists only of later inspections.

One limitation cannot be resolved from these two files: we do not know the target-extraction cutoff. In production, confirm that every row—especially the newest—had a complete 90-day observation window. The embargo protects validation from label look-ahead, but cannot repair incorrectly censored labels in the source data.

## 3. Inspection-time feature engineering and temporal split

In [ ]:
model_df = df.sort_values("inspected_at").reset_index(drop=True).copy()

# Cyclical encodings preserve recurring calendar/time relationships without
# treating December→January or 23:00→00:00 as large numeric jumps.
cycles = {
    "month": (model_df["inspected_at"].dt.month - 1, 12),
    "hour": (model_df["inspected_at"].dt.hour, 24),
    "dow": (model_df["inspected_at"].dt.dayofweek, 7),
}
for name, (values, period) in cycles.items():
    model_df[f"{name}_sin"] = np.sin(2 * np.pi * values / period)
    model_df[f"{name}_cos"] = np.cos(2 * np.pi * values / period)

categorical_features = ["site"]
numeric_features = [
    "vibration_rms", "bearing_temp_c", "ambient_temp_c", "meter_hours", "load_pct",
    "month_sin", "month_cos", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
]
features = categorical_features + numeric_features
target = "failed_90d"

test_start = model_df.loc[int(0.80 * len(model_df)), "inspected_at"]
train_label_cutoff = test_start - pd.Timedelta(days=90)
train = model_df[model_df["inspected_at"] < train_label_cutoff].copy()
test = model_df[model_df["inspected_at"] >= test_start].copy()
embargoed = model_df[
    (model_df["inspected_at"] >= train_label_cutoff) &
    (model_df["inspected_at"] < test_start)
].copy()

split_summary = pd.DataFrame({
    "rows": [len(train), len(embargoed), len(test)],
    "start": [train["inspected_at"].min(), embargoed["inspected_at"].min(), test["inspected_at"].min()],
    "end": [train["inspected_at"].max(), embargoed["inspected_at"].max(), test["inspected_at"].max()],
    "failures": [train[target].sum(), embargoed[target].sum(), test[target].sum()],
    "prevalence": [train[target].mean(), embargoed[target].mean(), test[target].mean()],
}, index=["train", "embargo (not used)", "final test"])
display(split_summary)

The embargoed rows are intentionally unused. At the start of the final test period, their 90-day outcomes would not yet be fully known.

## 4. Fit candidate models

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("site", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

candidate_models = {
    "Logistic regression": LogisticRegression(
        max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient boosting": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=10, max_features=0.7,
        class_weight="balanced", n_jobs=1, random_state=RANDOM_STATE,
    ),
}

def make_pipeline(model):
    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(model)),
    ])

def score_predictions(y_true, probability, top_fraction=0.10):
    y = np.asarray(y_true)
    p = np.asarray(probability)
    k = int(np.ceil(len(y) * top_fraction))
    top_idx = np.argsort(-p)[:k]
    prevalence = y.mean()
    top_precision = y[top_idx].mean()
    return {
        "ROC AUC": roc_auc_score(y, p),
        "Average precision": average_precision_score(y, p),
        "Brier score": brier_score_loss(y, p),
        "Mean predicted risk": p.mean(),
        "Top 10% precision": top_precision,
        "Top 10% recall": y[top_idx].sum() / y.sum(),
        "Top 10% lift": top_precision / prevalence,
    }

fitted = {}
predictions = {}
rows = []
for name, estimator in candidate_models.items():
    pipeline = make_pipeline(estimator)
    pipeline.fit(train[features], train[target])
    probability = pipeline.predict_proba(test[features])[:, 1]
    fitted[name] = pipeline
    predictions[name] = probability
    rows.append({"Model": name, **score_predictions(test[target], probability)})

results = pd.DataFrame(rows).set_index("Model").sort_values("Average precision", ascending=False)
display(results.style.format("{:.3f}"))
print(f"No-skill AP baseline (test prevalence): {test[target].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, probability in predictions.items():
    fpr, tpr, _ = roc_curve(test[target], probability)
    precision, recall, _ = precision_recall_curve(test[target], probability)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(test[target], probability):.3f})")
    axes[1].plot(recall, precision, label=f"{name} ({average_precision_score(test[target], probability):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=.5)
axes[0].set(title="ROC curve", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].axhline(test[target].mean(), color="black", linestyle="--", alpha=.5,
                label=f"No skill ({test[target].mean():.3f})")
axes[1].set(title="Precision–recall curve", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

ROC AUC describes ranking across both classes. Average precision is especially important here because failures are rare in the final period. Brier score and mean predicted risk reveal that class weighting can improve ranking while producing probabilities that are not calibrated; those raw probabilities must not be interpreted as literal failure chances without calibration on newer, fully matured data.

## 5. Stability across earlier forward-validation windows

In [ ]:
# Each window trains only on observations at least 90 days older than the
# validation start. These windows are diagnostic; the final test remains later in time.
fold_specs = [
    (0.50, 0.62),
    (0.65, 0.77),
]
cv_rows = []
for start_fraction, end_fraction in fold_specs:
    val_start = model_df.loc[int(start_fraction * len(model_df)), "inspected_at"]
    val_end = model_df.loc[int(end_fraction * len(model_df)), "inspected_at"]
    fold_train = model_df[model_df["inspected_at"] < val_start - pd.Timedelta(days=90)]
    fold_val = model_df[
        (model_df["inspected_at"] >= val_start) &
        (model_df["inspected_at"] < val_end)
    ]
    for name, estimator in candidate_models.items():
        pipeline = make_pipeline(estimator)
        pipeline.fit(fold_train[features], fold_train[target])
        p = pipeline.predict_proba(fold_val[features])[:, 1]
        metrics = score_predictions(fold_val[target], p)
        cv_rows.append({
            "Window start": val_start.date(),
            "Model": name,
            "Validation rows": len(fold_val),
            "Prevalence": fold_val[target].mean(),
            "ROC AUC": metrics["ROC AUC"],
            "Average precision": metrics["Average precision"],
            "Top 10% lift": metrics["Top 10% lift"],
        })

forward_results = pd.DataFrame(cv_rows)
display(forward_results.style.format({
    "Prevalence": "{:.3f}", "ROC AUC": "{:.3f}",
    "Average precision": "{:.3f}", "Top 10% lift": "{:.2f}",
}))

## 6. What drives the best deployable ranking model?

In [ ]:
best_name = results.index[0]
best_model = fitted[best_name]

# Held-out permutation importance measures AP loss when each original feature
# is shuffled. With few failures in the final test, treat the exact ordering as uncertain.
importance = permutation_importance(
    best_model, test[features], test[target], scoring="average_precision",
    n_repeats=8, random_state=RANDOM_STATE, n_jobs=1,
)
importance_df = (
    pd.DataFrame({
        "feature": features,
        "AP decrease mean": importance.importances_mean,
        "AP decrease std": importance.importances_std,
    })
    .sort_values("AP decrease mean", ascending=False)
)
display(importance_df)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = importance_df.sort_values("AP decrease mean")
ax.barh(plot_df["feature"], plot_df["AP decrease mean"], xerr=plot_df["AP decrease std"])
ax.axvline(0, color="black", linewidth=1)
ax.set(title=f"Permutation importance for {best_name}", xlabel="Decrease in held-out average precision")
plt.tight_layout()
plt.show()

## 7. Quantify the misleading effect of `parts_replaced`

In [ ]:
leaky_features = features + ["parts_replaced"]
leaky_numeric = numeric_features + ["parts_replaced"]
leaky_preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), leaky_numeric),
    ("site", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
leaky_model = Pipeline([
    ("preprocess", leaky_preprocessor),
    ("model", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
])
leaky_model.fit(train[leaky_features], train[target])
leaky_probability = leaky_model.predict_proba(test[leaky_features])[:, 1]
leaky_metrics = pd.Series(score_predictions(test[target], leaky_probability), name="Leaky model")
display(leaky_metrics.to_frame().style.format("{:.3f}"))

The leaky result is shown only as a warning. It is not evidence that a high-performing inspection-time system exists. Before any deployment, establish the exact timestamp at which `parts_replaced` becomes available; only use it if the business prediction is intentionally made after work-order closure.

## 8. Conclusion

In [ ]:
best = results.loc[best_name]
baseline = test[target].mean()
leaky_ap = average_precision_score(test[target], leaky_probability)

display(Markdown(f'''
### Answer: yes for risk ranking, not yet for autonomous pass/fail decisions

Using only inspection-time information, **{best_name}** achieved:

- **ROC AUC {best['ROC AUC']:.3f}** on the latest, embargoed holdout;
- **average precision {best['Average precision']:.3f}**, versus a no-skill baseline of **{baseline:.3f}**;
- **{best['Top 10% lift']:.1f}× lift** in the highest-risk 10% of inspections;
- the highest-risk 10% captured **{best['Top 10% recall']:.1%}** of failures, at **{best['Top 10% precision']:.1%}** precision.

That is meaningful ranking signal for prioritizing limited follow-up capacity, but precision remains modest because the latest-period failure rate is only {baseline:.1%}. The probability estimates are also not deployment-calibrated.

Adding `parts_replaced` raises AP to **{leaky_ap:.3f}**, but its timing and near-deterministic relationship with the target make that result leakage-prone and unsuitable for the stated inspection-time prediction.

### Recommended next steps

1. Verify that all labels have a full 90-day observation window and document the label cutoff.
2. Confirm prediction timing and permanently exclude every post-inspection/work-order outcome field.
3. Gather repeated inspections per asset plus asset type, age, maintenance history, prior anomalies, duty cycle, and sensor trends.
4. Prospectively score future inspections, monitor AP/lift and calibration by month/site, then select a threshold from maintenance capacity and failure costs.
'''))